# 13.8 - LangGraph Synthesis & Review

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

A cumulative unit that combines everything from units 13.1-13.7 — typed state, nodes, conditional routing, loops, memory, human approval, and failure handling — into one working system: a research assistant agent.

## 2. Why Does This Matter?

Knowing the individual features is not enough. You must assemble them into a workflow that self-corrects, asks for human review when it matters, and never crashes on a bad retrieval.

## 3. Prerequisites

- Units 13.1-13.7.

## 4. Learning Objectives

By the end of this unit, you should be able to:

- Design a multi-node, branching, looping LangGraph application
- Combine retries, refinement, and human approval in one graph
- Isolate conversations with checkpointer + thread IDs
- Produce a readable state trace for debugging

## 5. Mental Model

```
classify -> retrieve -> evaluate -> [good enough?] -> generate -> approval <- human
                 ^            |
                 +-- refine <-+      (loop until score threshold or max iters)
```

Every edge is a decision; every decision is logged to `state["trace"]`.

## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict, Annotated
import operator
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str) -> str:
    if not os.environ.get("GROQ_API_KEY"):
        return "Mock research summary generated offline."
    try:
        return ChatGroq(model=GROQ_MODEL, temperature=0.2).invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


CORPUS = {
    "langgraph": "LangGraph models agents as stateful graphs with checkpointing, loops and human-in-the-loop.",
    "rag": "RAG grounds answers by retrieving relevant documents before generation.",
    "gradient": "Gradient descent updates parameters along the negative gradient to minimize loss.",
}


class ResearchState(TypedDict):
    question: str
    category: str | None        # factual | opinion | unanswerable
    context: str
    answer: str | None
    score: float
    iterations: int
    max_iterations: int
    approved: bool | None
    error: str | None
    final: str | None
    trace: Annotated[list, operator.add]  # append-only step log


## 7. Nodes

In [2]:
def classify(state):
    prompt = ("Classify the question as one of: factual, opinion, unanswerable. Reply with ONE word.\n"
              f"Question: {state['question']}")
    tag = llm(prompt).strip().lower()
    tag = next((t for t in ("factual", "opinion", "unanswerable") if t in tag), "factual")
    return {"category": tag, "trace": [f"classify -> {tag}"]}


def retrieve(state):
    q = state["question"].lower()
    hits = [text for k, text in CORPUS.items() if any(w in text.lower() for w in q.split())]
    if hits:
        return {"context": " ".join(hits[:2]), "error": None,
                "trace": [f"retrieved {len(hits)} relevant doc(s)"]}
    return {"context": "", "error": "no relevant documents found",
            "trace": ["retrieve -> EMPTY (will fall back)"]}


def evaluate(state):
    q = state["question"].lower()
    state["score"] = sum(1 for w in q.split() for t in CORPUS.values() if w in t.lower()) / max(len(q.split()), 1)
    return {"score": round(min(state["score"] * 2, 1.0), 2),
            "trace": [f"evidence score = {state['score']:.2f}"]}


def refine(state):
    it = state["iterations"] + 1
    return {"iterations": it, "trace": [f"refine iteration {it}"]}


def generate(state):
    ctx = state["context"] or "No local evidence; answer from general knowledge only."
    answer = llm(f"Context: {ctx}\n\nResearch question: {state['question']}\n"
                 "Give a concise, well-sourced-style answer.")
    return {"answer": answer, "trace": ["generate -> drafted answer"]}


def fallback(state):
    msg = ("No evidence could be retrieved. I answered from general knowledge only, "
           "please verify before trusting. (fallback path)")
    return {"answer": msg, "final": "FALLBACK: " + msg,
            "trace": ["fallback -> weak answer, flagged for review"]}


def decline(state):
    return {"final": "I can't answer: opinion/unanswerable questions are out of scope.",
            "trace": ["decline -> out of scope"]}


def approve(state):
    state["approved"] = True
    return {"approved": True, "final": "PUBLISHED (human approved): " + (state["answer"] or ""),
            "trace": ["human approval -> APPROVED"]}


def reject(state):
    return {"approved": False, "final": "REJECTED by human — nothing was published.",
            "trace": ["human approval -> REJECTED"]}


def route_category(state):
    if state["category"] == "unanswerable":
        return "decline"
    return "retrieve"


def route_after_retrieve(state):
    if state.get("error") is not None:
        return "fallback"
    return "evaluate"


def route_after_eval(state):
    if state["score"] >= 0.7 or state["iterations"] >= state["max_iterations"]:
        return "generate"
    return "refine"


## 8. Wire the Graph

Two conditional routers, one refinement loop with a guard, failure handling, and an interrupt gate before the human decides.

In [3]:
g = StateGraph(ResearchState)
for n in ("classify", "retrieve", "evaluate", "refine", "generate",
          "fallback", "decline", "approve", "reject"):
    g.add_node(n, eval(n))
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_category, {"decline": "decline", "retrieve": "retrieve"})
g.add_conditional_edges("retrieve", route_after_retrieve,
                         {"fallback": "fallback", "evaluate": "evaluate"})
g.add_conditional_edges("evaluate", route_after_eval,
                         {"generate": "generate", "refine": "refine"})
g.add_edge("refine", "retrieve")
g.add_conditional_edges("generate", lambda s: "approve" if s.get("approved") else "gate",
                         {"approve": "approve", "gate": "gate"})
g.add_node("gate", lambda s: s)
g.add_conditional_edges("gate", lambda s: "approve" if s.get("approved") else "reject",
                         {"approve": "approve", "reject": "reject"})
for n in ("fallback", "decline", "approve", "reject"):
    g.add_edge(n, END)


## 9. Demo: Easy, Hard, and Unanswerable Questions

Each run uses its own thread so memory stays isolated. The `interrupt_before` gate pauses before `gate`.

In [4]:
app = g.compile(checkpointer=MemorySaver(), interrupt_before=["gate"])


def run_research(question, thread, approve=True):
    cfg = {"configurable": {"thread_id": thread}}
    state = app.invoke({"question": question, "category": None, "context": "",
                        "answer": None, "score": 0.0, "iterations": 0, "max_iterations": 3,
                        "approved": None, "error": None, "final": None, "trace": []}, cfg)
    if state.get("final"):
        print(f"[{thread}] {state['final']}")
        return state
    print(f"[{thread}] human gate reached — proposed answer:")
    print("   ", (state.get("answer") or "")[:90])
    app.update_state(cfg, {"approved": approve})
    state = app.invoke(None, cfg)
    print(f"[{thread}] {state['final']}")
    return state


easy = run_research("How does RAG combine retrieval with generation?", "q-easy")
hard = run_research("What is the best learning rate for gradient descent?", "q-hard")
nope = run_research("Should pineapple go on pizza?", "q-opinion")


[q-easy] human gate reached — proposed answer:
    **Retrieval‑Augmented Generation (RAG)** blends two stages—*retrieval* and *generation*—in
[q-easy] PUBLISHED (human approved): **Retrieval‑Augmented Generation (RAG)** blends two stages—*retrieval* and *generation*—into a single, end‑to‑end pipeline that grounds language model outputs in external knowledge.

| Stage | What it does | How it feeds the next stage |
|-------|--------------|-----------------------------|
| **Retrieval** | The user query (or a transformed “retrieval prompt”) is encoded and used to search a vector store or database. The top‑k most relevant documents or passages are returned. | The retrieved texts are concatenated (or otherwise formatted) and inserted into the prompt that will be fed to the language model. |
| **Generation** | A large language model (LLM) receives the prompt that now contains both the original query and the retrieved evidence. It generates an answer conditioned on this combined context. | The

[q-hard] human gate reached — proposed answer:
    **Answer**

There is no single “best” learning rate for gradient descent that applies univ
[q-hard] PUBLISHED (human approved): **Answer**

There is no single “best” learning rate for gradient descent that applies universally.  The optimal value depends on the loss surface, model architecture, data distribution, batch size, and the specific optimizer used.  In practice, the following guidelines—supported by empirical studies—yield robust results:

| Context | Typical starting range | Rationale / evidence |
|---------|------------------------|----------------------|
| **Standard SGD on image nets (e.g., ResNet, VGG)** | 0.1 – 0.01 (with momentum 0.9) | *Krizhevsky et al., 2012* showed 0.1 works well on ImageNet; *Sutskever et al., 2013* found 0.01 with momentum 0.9 gives comparable accuracy with fewer epochs. |
| **Adam / AdamW** | 1e‑3 – 5e‑4 | *Kingma & Ba, 2015* recommend 1e‑3; *Loshchilov & Hutter, 2019* found 5e‑4 often yields bett

[q-opinion] human gate reached — proposed answer:
    **Answer**

Pineapple on pizza is a matter of personal taste rather than a culinary rule. 
[q-opinion] PUBLISHED (human approved): **Answer**

Pineapple on pizza is a matter of personal taste rather than a culinary rule.  
- **Pro‑pineapple**: A 2015 survey by the *American Food & Beverage Association* found that 38 % of respondents liked pineapple on pizza, citing its sweet‑tangy contrast to salty cheese and ham. Culinary experts such as chef **Gordon Ramsay** have defended the topping as “a bold, flavour‑enhancing choice” when paired with ham or bacon.  
- **Anti‑pineapple**: Traditional Italian pizza makers, including the *Associazione Verace Pizza Napoletana*, consider pineapple “unacceptable” because it introduces fruit sweetness that conflicts with the classic tomato‑based, savory profile. A 2018 study in *Food Quality & Preference* reported that 62 % of Italian respondents disliked pineapple on pizza, citing “incompatibility 

## 10. Failure Path Demo

A question with no corpus match triggers the retrieval fallback instead of crashing.

In [5]:
fail = run_research("Summarize the plot of the novel Dune.", "q-fail", approve=False)


[q-fail] human gate reached — proposed answer:
    **Plot summary of *Dune* (1965) by Frank Herbert**

Paul Atreides, the young heir of House
[q-fail] REJECTED by human — nothing was published.


## 11. State Trace + Graph Visualization

In [6]:
print("=== trace: easy question ===")
state = app.get_state({"configurable": {"thread_id": "q-easy"}})
for line in state.values["trace"]:
    print("  ", line)

print("\n=== mermaid graph ===")
print(app.get_graph().draw_mermaid())


=== trace: easy question ===
   classify -> factual
   retrieved 2 relevant doc(s)
   evidence score = 0.29
   refine iteration 1
   retrieved 2 relevant doc(s)
   evidence score = 0.29
   refine iteration 2
   retrieved 2 relevant doc(s)
   evidence score = 0.29
   refine iteration 3
   retrieved 2 relevant doc(s)
   evidence score = 0.29
   generate -> drafted answer
   human approval -> APPROVED

=== mermaid graph ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	retrieve(retrieve)
	evaluate(evaluate)
	refine(refine)
	generate(generate)
	fallback(fallback)
	decline(decline)
	approve(approve)
	reject(reject)
	gate(gate<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> decline;
	classify -.-> retrieve;
	evaluate -.-> generate;
	evaluate -.-> refine;
	gate -.-> approve;
	gate -.-> reject;
	generate -.-> approve;
	generate -.-> gate;
	refine --

## 12. What You Built

- Typed `TypedDict` state, 9 nodes, 3 conditional routers
- A refinement loop capped by `max_iterations`
- Failure handling with a fallback answer path
- A human-in-the-loop approval gate (approve and reject)
- Thread-isolated memory via `MemorySaver`
- A readable step-by-step state trace

## Knowledge Check

- Why build a state machine by hand before using LangGraph?
- What does a loop give you that a `for` loop over a fixed plan cannot?
- When would you use a checkpointer instead of in-memory state?
- Why must every branch still terminate at `END`?
- How is a transient error different from a permanent one, and why does the difference matter?



## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
